# Rally cues on full matches (game-state dataset)

Extracts the rally cues of `scripts/extract_rally_cues.py` (court visible, players and their motion, ball detected and
its speed; 5 samples per second) from the 8 full broadcast matches of the game-state dataset
(masouduut94/volleyball_analytics; rally boundaries labelled with CVAT). Training and evaluation of the rally detector
run locally on these small CSV files (`scripts/eval_rallies.py`).

Runs on **Kaggle** (GPU, Internet on) with input `yoga0702/vball-models` (private: court and player weights).
Output: `/kaggle/working/cues/<match>.csv`.

In [ ]:
import glob, os, subprocess, zipfile
WORK = '/tmp/work'; OUT = '/kaggle/working/cues'
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
!git clone -q --depth 1 https://github.com/asigatchov/fast-volleyball-tracking-inference {WORK}/repo/external/fast-volleyball-tracking-inference
!mkdir -p {WORK}/repo/models && cp {WORK}/repo/external/fast-volleyball-tracking-inference/models/VballNetV4c_seq9_grayscale_*.onnx {WORK}/repo/models/vballnet_v4c.onnx
for f in glob.glob('/kaggle/input/**/*.pt', recursive=True):
    !cp "{f}" {WORK}/repo/models/
!pip -q install -e {WORK}/repo --no-deps
!pip -q install ultralytics onnxruntime-gpu gdown

In [ ]:
ZIP = f'{WORK}/gamestate.zip'
!gdown -q 1bhGTMQUIUPYMlwhNS4z6DwsnMaGKIBpn -O {ZIP}
z = zipfile.ZipFile(ZIP)
videos = sorted((n for n in z.namelist() if not n.endswith('.xml')), key=lambda n: int(n.split('.')[0]))
print(videos)

In [ ]:
for name in videos:
    out = f"{OUT}/{name.split('.')[0]}.csv"
    if os.path.exists(out):
        continue
    z.extract(name, WORK)
    !cd {WORK}/repo && python scripts/extract_rally_cues.py --video {WORK}/{name} --out {out} --device 0
    os.remove(f'{WORK}/{name}')
print(sorted(os.listdir(OUT)))